In [ ]:
from pathlib import Path
import json
import warnings
import sys
import pickle

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation
from visualization.src.utils import BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS, ARCHITECUTURE_FAMILY_COLORS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:

artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
results_dir = repo_dir / "extract_n_eval" / "results"
configs_dir = repo_dir / "extract_n_eval" / "cfgs"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"
assert results_dir.exists(), f"results directory {results_dir} does not exist!"
assert configs_dir.exists(), f"configs directory {configs_dir} does not exist!"

In [ ]:
layer_commitments_path = configs_dir / "layer_commitments.json"
with open(layer_commitments_path, "r") as f:
    layer_commitments = json.load(f)
layer_commitments = layer_commitments['deit_small_imagenet_full_seed-0']

In [ ]:
valid_models  = [
    'deit_small_imagenet_full_seed-0',
    'deit_large_imagenet_full_seed-0',
    'convnext_large_imagenet_full_seed-0',
    'resnet152_imagenet_full',
]

In [ ]:
df_proj = pd.read_csv(artifacts_dir / "scaling_results.csv")
df_proj = df_proj[df_proj['model_id'].isin(valid_models)]
df_proj

In [ ]:
VALID_FILENAMES = [
    "bs_fz.json",
    "bs_mh.json",
    "tvsd.json",
    "things_fmri.json",
    "things_eeg1.json",
    "things_eeg2.json",
    "things_meg.json",
    "nsd_func1pt8mm_individualROIs.json",
    "nsd_fsaverage_individualROIs.json",
    "nsd_nativesurface_individualROIs.json",
]
    
def get_results(results_dir: Path, leave: bool = True) -> pd.DataFrame:
    found_results_files = [file for file in results_dir.glob("**/*.json") if file.name in VALID_FILENAMES]

    all_results = []

    for file in tqdm(found_results_files, leave=leave):
        try:
            with open(file, 'r') as f:
                result = json.load(f)
        except Exception as e:
            print(f"Error loading {file}: {e}")
            continue
        all_results.extend(result["results"])
        
    df_all_results = pd.DataFrame(all_results)
    return df_all_results

In [ ]:
results_proj_dir = results_dir / "proj_feats_ablation"
result_folders = list(results_proj_dir.glob("dim_*-seed_*"))

# results_proj_30000_dir = results_dir / "proj_feats" / "deit_small_imagenet_full_seed-0"
# result_folders.append(results_proj_30000_dir)


proj_seed_results = []
for proj_seed_dir in tqdm(result_folders):
    if not proj_seed_dir.is_dir():
        continue

    proj, seed = proj_seed_dir.name.split("-")
    proj = int(proj.replace("dim_", ""))
    seed = int(seed.replace("seed_", ""))
    
    df_results = get_results(proj_seed_dir, leave=False)
    df_results["proj_dim"] = proj
    df_results["seed"] = seed
    proj_seed_results.append(df_results)
    
    
    

df_proj_ablation = pd.concat(proj_seed_results, ignore_index=True)

df_filtered = []
for benchmark_name, benchmark_rois in layer_commitments.items():
    for roi_name, layer in benchmark_rois.items():
        df_subset = df_proj_ablation[
            (df_proj_ablation['benchmark_name'] == benchmark_name) &
            (df_proj_ablation['roi'] == roi_name) &
            (df_proj_ablation['layer_name'] == layer['layer_name'])
        ]
        df_filtered.append(df_subset)

df_proj_ablation = pd.concat(df_filtered, ignore_index=True)
df_proj_ablation = df_proj_ablation.sort_values(by=['model_id', 'benchmark_name', 'roi', 'proj_dim', 'seed']).reset_index(drop=True)

df_proj_ablation

In [ ]:
df_proj_ablation[df_proj_ablation['benchmark_name'] == "tvsd"]

In [ ]:
results_full_dir = results_dir / "full_feats_committed2"
result_folders = list(results_full_dir.glob("*"))

# results_proj_30000_dir = results_dir / "proj_feats" / "deit_small_imagenet_full_seed-0"
# result_folders.append(results_proj_30000_dir)


full_results = []
for proj_seed_dir in tqdm(result_folders):
    if not proj_seed_dir.is_dir():
        continue
    
    df_results = get_results(proj_seed_dir, leave=False)
    full_results.append(df_results)
    
    
    

df_full = pd.concat(full_results, ignore_index=True)
df_full = df_full[df_full['model_id'].isin(valid_models)]

# df_filtered = []



df_full

In [ ]:
df_full[df_full.pearsonr_nc.isna()]

In [ ]:
# Remove specific problematic entries

df_proj_filtered = df_proj[
    ~((df_proj['model_id']=='resnet152_imagenet_full') & (df_proj['benchmark_name'] == 'tvsd') & (df_proj['roi'] == 'V1') ) &
    ~((df_proj['model_id']=='resnet152_imagenet_full') & (df_proj['benchmark_name'] == 'things_fmri') & (df_proj['subject'] == 'sub-02') & (df_proj['roi'] == 'whole_brain') ) &
    ~((df_proj['model_id']=='deit_large_imagenet_full_seed-0') & (df_proj['benchmark_name'] == 'things_fmri') & (df_proj['subject'] == 'sub-02') & (df_proj['roi'] == 'whole_brain') )
].copy()

# Remove rows with pearsonr_nc as NaN
df_proj_filtered = df_proj_filtered[~df_proj_filtered['pearsonr_nc'].isna()].copy()
df_full_filtered = df_full[~df_full['pearsonr_nc'].isna()].copy()

In [ ]:
df_full_filtered.shape, df_proj_filtered.shape

In [ ]:

metric_columns = [
    'cv_score',
    'r2',
    'evs',
    'mae',
    'mse',
    'pearsonr',
    'approx_exp_var',
    'noise_ceiling',
    'pearsonr_nc',
    'approx_exp_var_nc',
    'rsa_c_train',
    'rsa_ve_train',
    'cka_c_train',
    'cka_ve_train',
    'rsa_c_test',
    'rsa_ve_test',
    'cka_c_test',
    'cka_ve_test',
]
groping_columns = [
    'benchmark_name',
    'model_id',
    'layer_name',
    'layer_position',
    'roi',
    'proj_dim',
    # 'subject',
    'seed'  
]


data = df_full_filtered[df_full_filtered.model_id == 'deit_small_imagenet_full_seed-0'].copy()
data['proj_dim'] = 100000  # Assign a large number to represent full features
data['seed'] = 0  # Assign a seed value for consistency


df = pd.concat([
    df_proj_ablation, data
    ], ignore_index=True)


# df_proj_avg = df.groupby(groping_columns).agg({metric: np.nanmean for metric in metric_columns}).reset_index()
df_proj_avg = apply_hiearchical_aggregation(
    df=df,
    groupby_cols=groping_columns,
    agg_cols=metric_columns
)

df_proj_avg.sort_values('proj_dim')

In [ ]:
benchmark_names = {
    'bs_fz': 'FZ-EP',
    'bs_mh': 'MH-EP',
    'tvsd': 'TVSD-EP',
    'things_fmri': 'T-fMRI',
    'nsd_func1pt8mm_individualROIs': 'NSD-fMRI',
    'nsd': 'NSD-fMRI',
    'things_eeg1': 'T-EEG1',
    'things_eeg2': 'T-EEG2',
    'things_meg': 'T-MEG',
    'average': 'Average'
}

benchmark_order = {
    k: i for i, k in enumerate(benchmark_names.keys())
}

In [ ]:
import matplotlib as mpl
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 5.5), dpi=300)

data = df_proj_avg.copy()
data = data[data['benchmark_name'].isin([
    'tvsd',
    'bs_fz',
    'bs_mh',
    'things_fmri',
    'nsd_func1pt8mm_individualROIs',
])]


data['benchmark_order'] = data['benchmark_name'].map(benchmark_order)
data = data.sort_values('benchmark_order')
data['benchmark_name'] = data['benchmark_name'].map(benchmark_names)
data = data.sort_values(by=['benchmark_order', 'proj_dim'])

# ---- build a palette for ALL unique proj_dim values ----
dims = sorted(data['proj_dim'].unique())
dims_proj = [d for d in dims if d != 100_000]

cmap = mpl.cm.get_cmap("flare")
norm = mpl.colors.Normalize(
    vmin=min(dims_proj) if dims_proj else 0,
    vmax=max(dims_proj) if dims_proj else 1,
)

palette = {d: cmap(norm(d)) for d in dims_proj}
# choose a color for "No projection" (keep it distinct from the gradient)
palette[100_000] = (0.2, 0.2, 0.2, 1.0)

sns.barplot(
    data=data,
    x='benchmark_name',
    y='pearsonr_nc',
    hue='proj_dim', 
    palette=palette,         
    ax=ax
)

ax.set_ylabel("Noise-normalized Pearson r", fontsize=14, fontweight='bold')
ax.set_xlabel("Benchmark", fontsize=14, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

legend_handles = [
    mpatches.Patch(
        facecolor=palette[d],
        edgecolor="none",
        label=("No projection" if d == 100_000 else f"Proj dim = {d}")
    )
    for d in dims
]

if ax.legend_ is not None:
    ax.legend_.remove()

ax.legend(
    handles=legend_handles,
    title="Projection",
    loc="upper left",
    frameon=True,
)

ax.set_title("Effect of Random Feature Projections / DeiT-S", fontsize=16, fontweight='bold')

plt.tight_layout()

figures_dir = save_dir
base_filename = 'random_feature_projections_dims'
formats = ['pdf', 'png', 'svg']
save_figs(fig, figures_dir, base_filename, formats=formats)

In [ ]:
data

In [ ]:
model_id_map = {
    'deit_small_imagenet_full_seed-0': 'DeiT-S',
    'deit_large_imagenet_full_seed-0': 'DeiT-L',
    'convnext_large_imagenet_full_seed-0': 'ConvNeXt-L',
    'resnet152_imagenet_full': 'ResNet-152',
}

model_order = {
    k: i for i, k in enumerate(model_id_map.keys())
}

In [ ]:
df_full_filtered['proj_type'] = 'No projection'
df_proj_filtered['proj_type'] = '30000-dimensional random projection'

df = pd.concat([df_proj_filtered, df_full_filtered], ignore_index=True)

df['model_order'] = df['model_id'].map(model_order)
df['data_order'] = df['benchmark_name'].map(benchmark_order)
df = df.sort_values(by=['model_order', 'data_order'])

df['benchmark_name'] = df['benchmark_name'].map(benchmark_names)

fig, axes = plt.subplots(2, 2, figsize=(20, 10), dpi=300, sharey=True)


for ax, model_id in zip(axes.flatten(), valid_models):
    data = df[df['model_id'] == model_id].copy()
    sns.barplot(
        data=data,
        x='benchmark_name',
        y='pearsonr_nc',
        hue='proj_type',
        ax=ax
    )
    ax.set_title(model_id_map[model_id], fontsize=16, fontweight='bold')
    ax.set_xlabel('Benchmark', fontsize=14, fontweight='bold')
    ax.set_ylabel("Noise ceiled Pearson's r", fontsize=14, fontweight='bold')
    ax.legend(title='Feature Type', fontsize=12, title_fontsize=12)
    ax.spines[['top', 'right']].set_visible(False)
    
fig.tight_layout()


figures_dir = save_dir
base_filename = 'random_feature_projections_architectures'
formats = ['pdf', 'png', 'svg']
save_figs(fig, figures_dir, base_filename, formats=formats)